In [ ]:
import jax
import jax.numpy as jnp
import jax.scipy as jsp
import matplotlib.pyplot as plt
import tinygp
import blackjax


from jax.flatten_util import ravel_pytree
from tinygp.helpers import dataclass

jax.config.update("jax_enable_x64", True)


# -----------------------
# LMC kernel
# -----------------------
@dataclass
class LMCKernel(tinygp.kernels.Kernel):
    kernels: tuple[tinygp.kernels.Kernel, ...]  # length I
    Phi: jnp.ndarray  # (I, V)

    def evaluate(self, X1, X2):
        x1, v1 = X1
        x2, v2 = X2
        w1 = self.Phi[:, v1]  # (I,)
        w2 = self.Phi[:, v2]  # (I,)
        k_vals = jnp.stack([k.evaluate(x1, x2) for k in self.kernels])  # (I,)
        return jnp.sum(w1 * w2 * k_vals)


# -----------------------
# Helpers
# -----------------------
LOG2PI = jnp.log(2.0 * jnp.pi)


def stdnorm_logpdf(z):
    # log N(z; 0, 1)
    return -0.5 * (z * z + LOG2PI)


def log_beta_binomial(y, n, a, b):
    log_binom = (
        jsp.special.gammaln(n + 1.0)
        - jsp.special.gammaln(y + 1.0)
        - jsp.special.gammaln(n - y + 1.0)
    )
    return log_binom + jsp.special.betaln(y + a, n - y + b) - jsp.special.betaln(a, b)


# -----------------------
# Synthetic data (multi-output BetaBinomial)
# -----------------------
key = jax.random.key(0)

S = 20
S_test = 200
I = 2
V = 15

x = jnp.linspace(0.0, 10.0, S)
x_test = jnp.linspace(-1.0, 11.0, S_test)

rho_true = 1.5
kernels = tuple(tinygp.kernels.Matern32(scale=rho_true) for _ in range(I))

key, kPhi = jax.random.split(key)
Phi_true = 0.7 * jax.random.normal(kPhi, (I, V))

sigma_true = 1.0
kappa_true = 25.0

x_obs = jnp.repeat(x, V)
v_obs = jnp.tile(jnp.arange(V, dtype=jnp.int32), S)
X = (x_obs, v_obs)
N = x_obs.shape[0]

kernel_true = LMCKernel(kernels=kernels, Phi=Phi_true) * (sigma_true**2)
K = kernel_true(X, X) + 1e-6 * jnp.eye(N)

key, kf, kb, ky = jax.random.split(key, 4)
f_true = jax.random.multivariate_normal(kf, mean=jnp.zeros(N), cov=K)
p_true = jax.nn.sigmoid(f_true)

n_obs = jnp.full((N,), 50, dtype=jnp.int32)
alpha_true = p_true * kappa_true
beta_true = (1.0 - p_true) * kappa_true

q = jax.random.beta(kb, alpha_true, beta_true)
y_obs = jax.random.binomial(ky, n=n_obs, p=q).astype(jnp.int32)
rate = (y_obs / n_obs).reshape(S, V)

for v in range(V):
    plt.figure()
    plt.scatter(x, rate[:, v], label=f"observed rate (v={v})")
    plt.ylim(-0.05, 1.05)
    plt.xlabel("x")
    plt.ylabel(f"p (output {v})")
    plt.title(f"GP-LMC + BetaBinomial (BlackJAX) — output {v}")
    plt.legend()
    plt.tight_layout()

plt.show()

In [ ]:
# -----------------------
# Log-posterior (PyTree)
# -----------------------
def logprob(position):
    f = position["f"]
    Phi = position["Phi"]

    rho = jax.nn.softplus(position["raw_rho"]) + 1e-6
    sigma = jax.nn.softplus(position["raw_sigma"]) + 1e-6
    kappa = jax.nn.softplus(position["raw_kappa"]) + 1e-6

    lp = 0.0
    # simple priors in raw space
    lp += stdnorm_logpdf(position["raw_rho"])
    lp += stdnorm_logpdf(position["raw_sigma"])
    lp += stdnorm_logpdf(position["raw_kappa"])
    lp += jnp.sum(stdnorm_logpdf(Phi))

    # GP prior: f ~ N(0, K_theta)
    base_kernels = tuple(tinygp.kernels.Matern32(scale=rho) for _ in range(I))
    kern = LMCKernel(kernels=base_kernels, Phi=Phi) * (sigma**2)
    Ktheta = kern(X, X) + 1e-6 * jnp.eye(N)
    L = jnp.linalg.cholesky(Ktheta)

    alpha = jsp.linalg.solve_triangular(L, f, lower=True)
    lp += -0.5 * jnp.dot(alpha, alpha)  # quadratic
    lp += -jnp.sum(jnp.log(jnp.diag(L)))  # -0.5 logdet
    lp += -0.5 * N * LOG2PI

    # BetaBinomial likelihood with p = sigmoid(f)
    p = jax.nn.sigmoid(f)
    p = jnp.clip(p, 1e-6, 1.0 - 1e-6)

    a = p * kappa
    b = (1.0 - p) * kappa

    lp += jnp.sum(
        log_beta_binomial(y_obs.astype(jnp.float64), n_obs.astype(jnp.float64), a, b)
    )
    return lp


# -----------------------
# Run NUTS (flattened)
# -----------------------
key, kinit, ksample = jax.random.split(key, 3)

position0 = {
    "f": 0.01 * jax.random.normal(kinit, (N,)),
    "Phi": 0.1 * jax.random.normal(kinit, (I, V)),
    "raw_rho": jnp.array(0.0),
    "raw_sigma": jnp.array(0.0),
    "raw_kappa": jnp.array(jnp.log(jnp.expm1(10.0))),  # ~softplus^-1(10)
}

pos0_flat, unravel = ravel_pytree(position0)
D = pos0_flat.size


def logprob_flat(z_flat):
    return logprob(unravel(z_flat))


step_size = 1e-3
inv_mass = jnp.ones((D,))  # diagonal metric

nuts = blackjax.nuts(logprob_flat, step_size, inv_mass)
state = nuts.init(pos0_flat)


@jax.jit
def one_step(state, key):
    state, info = nuts.step(key, state)
    return state, (state.position, info.acceptance_rate)


num_warmup = 400
num_samples = 600

keys = jax.random.split(ksample, num_warmup + num_samples)
state, _ = jax.lax.scan(lambda st, k: one_step(st, k), state, keys[:num_warmup])
state, (positions_flat, acc_rates) = jax.lax.scan(  # ### FIX: name it positions_flat
    lambda st, k: one_step(st, k), state, keys[num_warmup:]
)

print("mean acceptance:", float(jnp.mean(acc_rates)))

# ### FIX: unflatten samples back to PyTree dicts
positions = jax.vmap(unravel)(positions_flat)


# -----------------------
# Posterior predictive for success-rate p(x,v)
# -----------------------
x_t = jnp.repeat(x_test, V)
v_t = jnp.tile(jnp.arange(V, dtype=jnp.int32), S_test)
X_test = (x_t, v_t)


def predict_p_from_sample(pos, key):
    Phi = pos["Phi"]
    rho = jax.nn.softplus(pos["raw_rho"]) + 1e-6
    sigma = jax.nn.softplus(pos["raw_sigma"]) + 1e-6

    base_kernels = tuple(tinygp.kernels.Matern32(scale=rho) for _ in range(I))
    kern = LMCKernel(kernels=base_kernels, Phi=Phi) * (sigma**2)

    gp_prior = tinygp.GaussianProcess(kern, X, diag=1e-6)
    _, gp_cond = gp_prior.condition(pos["f"], X_test=X_test)

    f_test = gp_cond.sample(key)  # (S_test*V,)
    return jax.nn.sigmoid(f_test).reshape(S_test, V)


thin = 10
positions_thin = jax.tree.map(lambda a: a[::thin], positions)
M = positions_thin["raw_rho"].shape[0]

# ### FIX: advance RNG for prediction keys
key, kpred = jax.random.split(key)
keys_pred = jax.random.split(kpred, M)

p_samps = jax.vmap(predict_p_from_sample)(positions_thin, keys_pred)  # (M, S_test, V)

p_med = jnp.median(p_samps, axis=0)
p_lo = jnp.quantile(p_samps, 0.1, axis=0)
p_hi = jnp.quantile(p_samps, 0.9, axis=0)

rate = (y_obs / n_obs).reshape(S, V)

for v in range(V):
    plt.figure()
    plt.scatter(x, rate[:, v], label=f"observed rate (v={v})")
    plt.plot(x_test, p_med[:, v], label="posterior median p(x)")
    plt.fill_between(x_test, p_lo[:, v], p_hi[:, v], alpha=0.2, label="80% band")
    plt.ylim(-0.05, 1.05)
    plt.xlabel("x")
    plt.ylabel(f"p (output {v})")
    plt.title(f"GP-LMC + BetaBinomial (BlackJAX) — output {v}")
    plt.legend()
    plt.tight_layout()

plt.show()

In [ ]:
import jax.numpy as jnp
import tinygp


class LMCKernel(tinygp.kernels.Kernel):
    # kernels: length-I collection of scalar kernels k_i(x,x')
    kernels: tuple[tinygp.kernels.Kernel, ...]
    # Phi: shape (I, V). If you think of f(x)=Phi^T u(x), this matches your dims.
    Phi: jnp.ndarray

    def evaluate(self, X1, X2):
        x1, v1 = X1  # v1 is an integer task/output id
        x2, v2 = X2

        # weights for the two outputs (shape: (I,))
        w1 = self.Phi[:, v1]
        w2 = self.Phi[:, v2]

        # latent kernel values (shape: (I,))
        k_vals = jnp.stack([k.evaluate(x1, x2) for k in self.kernels])

        return jnp.sum(w1 * w2 * k_vals)

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import tinygp


from tinygp.helpers import dataclass  # <-- important


@dataclass
class LMCKernel(tinygp.kernels.Kernel):
    kernels: tuple[tinygp.kernels.Kernel, ...]  # length I
    Phi: jnp.ndarray  # (I, V)

    def evaluate(self, X1, X2):
        # v is an idx indicating the output chanel associated
        # with the x input
        x1, v1 = X1
        x2, v2 = X2

        w1 = self.Phi[:, v1]  # (I,)
        w2 = self.Phi[:, v2]  # (I,)

        k_vals = jnp.stack([k.evaluate(x1, x2) for k in self.kernels])  # (I,)
        return jnp.sum(w1 * w2 * k_vals)


# --- 2) Synthetic data sampled from the same model (so the GP is well-specified)
key = jax.random.key(0)

S = 10
S_test = 200
I = 2
V = 4

x = jnp.linspace(0.0, 10.0, S)
x_test = jnp.linspace(-1.0, 11.0, S_test)

# Latent kernels k_i, there are I kernels
kernels = [tinygp.kernels.Matern32(scale=1.2)] * I

# True mixing matrix Phi (I, V)
key, kPhi = jax.random.split(key)
Phi_true = 0.8 * jax.random.normal(kPhi, (I, V))

# Sample independent latent GPs u_i(x): shape (S, I)
u = []
for i in range(I):
    key, kz = jax.random.split(key)
    K = kernels[i](x, x) + 1e-6 * jnp.eye(S)
    L = jnp.linalg.cholesky(K)
    u_i = L @ jax.random.normal(kz, (S,))
    u.append(u_i)
u = jnp.stack(u, axis=1)  # (S, I)

# Multi-output signal f(x) = u(x) @ Phi: (S, V)
f_clean = u @ Phi_true

# Per-output noise
noise_per_output = jnp.array([0.15] * V)  # (V,)
key, kn = jax.random.split(key)
y = f_clean + jax.random.normal(kn, (S, V)) * noise_per_output[None, :]

# Pack data as (x_obs, v_obs), y_obs
x_obs = jnp.repeat(x, V)  # (S*V,)
v_obs = jnp.tile(jnp.arange(V, dtype=jnp.int32), S)  # (S*V,)
y_obs = y.reshape(-1)  # (S*V,)

diag = jnp.tile(noise_per_output**2, S)  # (S*V,)

# --- 3) Fit/condition GP (here we use the true Phi so you can test correctness)
kernel = LMCKernel(kernels=kernels, Phi=Phi_true)
gp = tinygp.GaussianProcess(kernel, (x_obs, v_obs), diag=diag)

# Condition at test points for all outputs
x_t = jnp.repeat(x_test, V)
v_t = jnp.tile(jnp.arange(V, dtype=jnp.int32), S_test)

# tinygp returns (conditional_mean_at_test, conditional_gp_object)
logp, gp_cond = gp.condition(y_obs, X_test=(x_t, v_t))

mu_test = gp_cond.loc.reshape(S_test, V)
std_test = jnp.sqrt(gp_cond.variance).reshape(S_test, V)

# Posterior marginal variances at test points (diagonal)
var_test = gp_cond.variance.reshape(S_test, V)
std_test = jnp.sqrt(var_test)

# --- 4) Plots (one figure per output; no subplots; no explicit colors)
for v in range(V):
    plt.figure()
    plt.scatter(x, y[:, v], label=f"observations (v={v})")
    plt.plot(x_test, mu_test[:, v], label="posterior mean")
    plt.fill_between(
        x_test,
        mu_test[:, v] - std_test[:, v],
        mu_test[:, v] + std_test[:, v],
        alpha=0.2,
        label="±1σ",
    )
    plt.xlabel("x")
    plt.ylabel(f"y[{v}]")
    plt.title(f"LMC multi-output GP (output {v})")
    plt.legend()
    plt.tight_layout()

plt.show()

## YO

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax

jax.config.update("jax_enable_x64", True)

from tinygp import GaussianProcess
from tinygp import kernels

kernel = kernels.ExpSquared(scale=1.5)
# Let's make up some input coordinates (sorted for plotting purposes)
X = np.sort(np.random.default_rng(1).uniform(0, 10, 100))

gp = GaussianProcess(kernel, X)

In [ ]:
def mean_function(params, X):
    mod = jnp.exp(-0.5 * jnp.square((X - params["loc"]) / jnp.exp(params["log_width"])))
    beta = jnp.array([1, mod])
    return params["amps"] @ beta


def multioutput_gp(params):
    kernel = jnp.exp(params["log_gp_amp"]) * kernels.Matern52(
        jnp.exp(params["log_gp_scale"])
    )
    return GaussianProcess(kernel, X, diag=jnp.exp(params["log_gp_diag"]))


@jax.jit
def loss_v2(params):
    gp = multioutput_gp(params)
    return -gp.log_probability(y - vmapped_mean_function(params, X))

In [ ]:
y = gp.sample(jax.random.PRNGKey(4), shape=(5,))
plt.plot(X, y.T, color="k", lw=0.5)
plt.xlabel("x")
plt.ylabel("sampled observations")
_ = plt.title("exponential squared kernel")